In [ ]:
# ==============================================================================
# 1. INSTALLATIONS & IMPORTS
# ==============================================================================
!pip install torch torchvision transformers accelerate pillow
!pip install evaluate rouge-score nltk
import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6533263304175787feab7b3e19b130a4ae531a9585ea3f6945e41cbf5c501995
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


True

In [ ]:
# ==============================================================================
# 2. IMPORTS AND GLOBAL SETUP
# ==============================================================================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from evaluate import load
import seaborn as sns

In [ ]:
# ==============================================================================
# 3. GLOBAL SETUP
# ==============================================================================
# Define test parameters (Change to see the effect of other keep ratios or pruning methods)
MAX_NEW_TOKENS = 40
KEEP_RATIOS = [0.8, 0.6, 0.4]
PRUNING_METHODS = ['random']
NUM_TIMING_TRIALS = 10 #Change this if you don't want test runs for the baseline

# Image list (Add other images if needed)
IMAGE_DATA = [
    {"url": "https://huggingface.co/spaces/HuggingFaceTB/SmolVLM-256M-Demo/resolve/main/example_images/rococo.jpg", "name": "Rococo Portrait"},
    {"url": "https://huggingface.co/spaces/HuggingFaceTB/SmolVLM-256M-Demo/resolve/main/example_images/newyork.jpg", "name": "New York"},
    {"url": "https://huggingface.co/spaces/HuggingFaceTB/SmolVLM-256M-Demo/resolve/main/example_images/examples_invoice.png", "name": "Invoice Example"},
    {"url": "https://huggingface.co/spaces/HuggingFaceTB/SmolVLM-256M-Demo/resolve/main/example_images/examples_weather_events.png", "name": "Weather Events"},
    {"url": "https://huggingface.co/spaces/HuggingFaceTB/SmolVLM-256M-Demo/resolve/main/example_images/campeones.jpg", "name": "Campeones"},
    {"url": "https://picsum.photos/id/237/800/600.jpg", "name": "Dog (placeholder)"},
    {"url": "https://picsum.photos/id/1025/800/600.jpg", "name": "Mountain Lake"},
    {"url": "https://picsum.photos/id/1062/800/600.jpg", "name": "Forest Path"},
    {"url": "https://picsum.photos/id/1074/800/600.jpg", "name": "Snowy Trees"},
    {"url": "https://picsum.photos/id/1080/800/600.jpg", "name": "Beach Sunset"}
]

In [ ]:
# ==============================================================================
# 4. MODEL SETUP
# ==============================================================================

# Note: This section was copied from the HuggingFace website (https://huggingface.co/blog/smolvlm)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model.eval()
print("SmolVLM model loaded successfully.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

SmolVLM model loaded successfully.


In [ ]:
# ==============================================================================
# 5. PRUNING FUNCTIONS (Updated: Ensure index order and timing is outside)
# ==============================================================================

def prune_tokens(input_embeds, keep_ratio, method):

    with torch.no_grad():
        batch_size, seq_len, embed_dim = input_embeds.shape
        num_keep = int(keep_ratio * seq_len)
        # If block is defined so other pruning methods can be added fo future
        if method == 'random':
            # Random importance
            importance = torch.rand(batch_size, seq_len, device=input_embeds.device)
        else:
            raise ValueError("Invalid pruning method.")

        # Get the indices of the 'num_keep' most important tokens
        _, topk_idx = torch.topk(importance, num_keep, dim=1)

        # Sort the indices to preserve the original sequential order (To make sure is does not effect inference time)
        topk_idx, _ = torch.sort(topk_idx, dim=1)

        # Gather the embeddings using the sorted indices
        pruned_embeds = torch.gather(
            input_embeds, 1,
            topk_idx.unsqueeze(-1).expand(-1, -1, embed_dim)
        )

        # Ensure memory is contiguous
        pruned_embeds = pruned_embeds.contiguous()

    return pruned_embeds, num_keep

def run_pruning_test(model, inputs, processor, keep_ratio, method, num_trials=NUM_TIMING_TRIALS):

    global MAX_NEW_TOKENS

    # Note: Pruning time itself is not included in the measurement
    input_embeds = model.get_input_embeddings()(inputs["input_ids"])
    pruned_embeds, num_keep = prune_tokens(input_embeds, keep_ratio, method)

    inputs_pruned = {**inputs}
    inputs_pruned.pop("input_ids", None)
    inputs_pruned["inputs_embeds"] = pruned_embeds

    # Benchmark the inference time using explicit CUDA synchronization
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    time_measurements = []

    # Warm-up run to initialize GPU kernels
    model.generate(**inputs_pruned, max_new_tokens=2, do_sample=False)

    for _ in range(num_trials):
        start_event.record()
        _ = model.generate(**inputs_pruned, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        end_event.record()
        torch.cuda.synchronize()
        time_measurements.append(start_event.elapsed_time(end_event) / 1000)

    time_pruned = np.mean(time_measurements)

    # Perform the actual generation
    generated_ids = model.generate(**inputs_pruned, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Return generated text and average time
    return text, time_pruned

In [ ]:
# ==============================================================================
# 6. MAIN TEST LOOP AND DATA COLLECTION (Updated: Prints time for each run)
# ==============================================================================
full_results = []

# Other metrics can be added
metrics = {'rouge': load('rouge'), 'meteor': load('meteor')}

#Change the text promp here if needed.
prompt_messages = [
    {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe the image."}]}
]

for img_data in IMAGE_DATA:
    print(f"\n=======================================================")
    print(f"STARTING IMAGE: {img_data['name']}")
    print(f"=======================================================")

    # Input Setup
    image = load_image(img_data['url']).resize((224, 224))
    prompt = processor.apply_chat_template(prompt_messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(device)

    # Run BASELINE (100%) for this image to get t_base
    print("--- BASELINE (100% Keep) ---")
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    time_measurements = []

    # Warm-up run
    model.generate(**inputs, max_new_tokens=2, do_sample=False)

    for _ in range(NUM_TIMING_TRIALS):
        start_event.record()
        _ = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        end_event.record()
        torch.cuda.synchronize()
        time_measurements.append(start_event.elapsed_time(end_event) / 1000)

    t_base = np.mean(time_measurements)

    # Run the generation one last time to get the output text for metrics
    baseline_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    baseline_text = processor.batch_decode(baseline_ids, skip_special_tokens=True)[0]
    print(f"Baseline Time: {t_base:.4f}s (Average of {NUM_TIMING_TRIALS} runs)")

    # Store baseline result
    full_results.append({
        'Image': img_data['name'], 'Keep Ratio': 1.0, 'Method': 'Baseline',
        'Output': baseline_text, 'Time (s)': t_base, 't_base': t_base
    })

    # Run Pruning Combinations
    for ratio in KEEP_RATIOS:
        for method in PRUNING_METHODS:
            output, time = run_pruning_test(
                model, inputs, processor, keep_ratio=ratio, method=method
            )

            print(f"\nTest: {method.upper()} @ {ratio*100:.0f}% (Average time: {time:.4f})")

            full_results.append({
                'Image': img_data['name'], 'Keep Ratio': ratio, 'Method': method.upper(),
                'Output': output, 'Time (s)': time, 't_base': t_base
            })

df_full = pd.DataFrame(full_results)
print("\n--- All tests complete. Calculating metrics and speedup... ---")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



STARTING IMAGE: Rococo Portrait
--- BASELINE (100% Keep) ---
Baseline Time: 2.2023s (Average of 10 runs)

Test: RANDOM @ 80% (Average time: 1.9734)

Test: RANDOM @ 60% (Average time: 2.0950)

Test: RANDOM @ 40% (Average time: 2.2609)

STARTING IMAGE: New York
--- BASELINE (100% Keep) ---
Baseline Time: 2.3611s (Average of 10 runs)

Test: RANDOM @ 80% (Average time: 2.4280)

Test: RANDOM @ 60% (Average time: 0.7111)

Test: RANDOM @ 40% (Average time: 2.0625)

STARTING IMAGE: Invoice Example
--- BASELINE (100% Keep) ---
Baseline Time: 2.5626s (Average of 10 runs)

Test: RANDOM @ 80% (Average time: 2.4950)

Test: RANDOM @ 60% (Average time: 2.4966)

Test: RANDOM @ 40% (Average time: 2.1589)

STARTING IMAGE: Weather Events
--- BASELINE (100% Keep) ---
Baseline Time: 2.6496s (Average of 10 runs)

Test: RANDOM @ 80% (Average time: 2.0697)

Test: RANDOM @ 60% (Average time: 0.8069)

Test: RANDOM @ 40% (Average time: 1.9037)

STARTING IMAGE: Campeones
--- BASELINE (100% Keep) ---
Baseline Tim

In [ ]:
# ==============================================================================
# 7. METRIC CALCULATION AND AGGREGATION
# ==============================================================================

# Note: Since Random Pruning is being used, results may fluctuate in every run

# Calculate Metrics for every single run
df_full['ROUGE-L Score'] = 0.0
df_full['METEOR Score'] = 0.0

# Calculate Speedup Percentage
df_full['Speedup (%)'] = 100 * (1 - df_full['Time (s)'] / df_full['t_base'])
df_full.loc[df_full['Method'] == 'Baseline', 'Speedup (%)'] = 0.0

for index, row in df_full.iterrows():

    if row['Method'] != 'Baseline':
        # Find the Baseline text for the current image
        baseline_row = df_full[(df_full['Image'] == row['Image']) & (df_full['Method'] == 'Baseline')].iloc[0]
        reference_text = baseline_row['Output']

        # Prepare candidates and references
        references = [reference_text]
        candidates = [row['Output']]

        # ROUGE-L
        rouge_result = metrics['rouge'].compute(predictions=candidates, references=references)
        df_full.loc[index, 'ROUGE-L Score'] = rouge_result['rougeL']

        # METEOR
        meteor_result = metrics['meteor'].compute(predictions=candidates, references=references)
        df_full.loc[index, 'METEOR Score'] = meteor_result['meteor']

# Averaging all runs
df_pruned_only = df_full[df_full['Method'] != 'Baseline']

df_avg = df_pruned_only.groupby(['Keep Ratio', 'Method']).agg(
    Avg_Time=('Time (s)', 'mean'),
    Avg_Speedup=('Speedup (%)', 'mean'),
    Avg_ROUGE_L=('ROUGE-L Score', 'mean'),
    Avg_METEOR=('METEOR Score', 'mean')
).reset_index()

# Final Formatting and Display
df_avg['Keep Ratio'] = (df_avg['Keep Ratio'] * 100).astype(int).astype(str) + '%'
df_avg['Avg. Speedup'] = df_avg['Avg_Speedup'].round(1).astype(str) + '%'
df_avg['Avg. Time (s)'] = df_avg['Avg_Time'].round(3)
df_avg['Avg. ROUGE-L'] = (df_avg['Avg_ROUGE_L'] * 100).round(2).astype(str) + '%'
df_avg['Avg. METEOR'] = df_avg['Avg_METEOR'].round(4)


df_final_table = df_avg[[
    'Keep Ratio',
    'Method',
    'Avg. Time (s)',
    'Avg. Speedup',
    'Avg. ROUGE-L',
    'Avg. METEOR'
]].sort_values(by=['Keep Ratio', 'Method'], ascending=[False, True])

print("\n\n=======================================================")
print("  FINAL AGGREGATED RESULTS (Average Across All Images)")
print("=======================================================")
print(df_final_table.to_markdown(index=False))



  FINAL AGGREGATED RESULTS (Average Across All Images)
| Keep Ratio   | Method   |   Avg. Time (s) | Avg. Speedup   | Avg. ROUGE-L   |   Avg. METEOR |
|:-------------|:---------|----------------:|:---------------|:---------------|--------------:|
| 80%          | RANDOM   |           1.904 | 15.4%          | 28.7%          |        0.2475 |
| 60%          | RANDOM   |           1.735 | 21.1%          | 20.93%         |        0.1894 |
| 40%          | RANDOM   |           1.957 | 11.8%          | 24.48%         |        0.2353 |


In [ ]:
# ==============================================================================
# 8. PLOT THE AGGREGATED RESULTS (REFINED)
# ==============================================================================

df_plot = df_final_table.copy()

# Convert Keep Ratio to float for x-axis sorting
df_plot['Keep Ratio (Value)'] = df_plot['Keep Ratio'].str.replace('%', '').astype(int) / 100

# Convert Avg. Speedup to float
df_plot['Avg. Speedup (%)'] = df_plot['Avg. Speedup'].str.replace('%', '').astype(float)

# Convert Avg. ROUGE-L to float
df_plot['Avg. ROUGE-L (%)'] = df_plot['Avg. ROUGE-L'].str.replace('%', '').astype(float)

# Convert Avg. METEOR to float if needed
if df_plot['Avg. METEOR'].dtype == 'object':
    df_plot['Avg. METEOR (Score)'] = df_plot['Avg. METEOR'].astype(float)
else:
    df_plot['Avg. METEOR (Score)'] = df_plot['Avg. METEOR']


# Set the theme for consistent styling
sns.set_theme(style="whitegrid")

# --- Plot 1: Average Speedup vs. Keep Ratio ---
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 6))
ax1.set_title('Average Speedup from Random Token Pruning', fontsize=14)

sns.lineplot(
    data=df_plot,
    x='Keep Ratio (Value)',
    y='Avg. Speedup (%)',
    marker='o',
    markersize=10,
    color='tab:blue',
    ax=ax1,
)

ax1.set_xlabel('Keep Ratio (Fraction of Tokens Kept)')
ax1.set_ylabel('Average Speedup (%) (Positive = Faster)')
ax1.set_xticks(df_plot['Keep Ratio (Value)'].unique())
ax1.set_xticklabels([f"{int(r*100)}%" for r in df_plot['Keep Ratio (Value)'].unique()])

plt.tight_layout()

# --- Plot 2: Accuracy (ROUGE-L) vs. Keep Ratio ---
fig2, ax2 = plt.subplots(1, 1, figsize=(8, 6))
ax2.set_title('Average Accuracy (ROUGE-L) from Random Token Pruning', fontsize=14)

sns.lineplot(
    data=df_plot,
    x='Keep Ratio (Value)',
    y='Avg. ROUGE-L (%)',
    marker='s',
    markersize=10,
    color='tab:orange',
    ax=ax2
)

ax2.set_xlabel('Keep Ratio (Fraction of Tokens Kept)')
ax2.set_ylabel('Average ROUGE-L Score (%)')
ax2.set_xticks(df_plot['Keep Ratio (Value)'].unique())
ax2.set_xticklabels([f"{int(r*100)}%" for r in df_plot['Keep Ratio (Value)'].unique()])

plt.tight_layout()

# --- Plot 3: Accuracy (METEOR) vs. Keep Ratio ---
fig3, ax3 = plt.subplots(1, 1, figsize=(8, 6))
ax3.set_title('Average Accuracy (METEOR) from Random Token Pruning', fontsize=14)

sns.lineplot(
    data=df_plot,
    x='Keep Ratio (Value)',
    y='Avg. METEOR (Score)',
    marker='^',
    markersize=10,
    color='tab:green',
    ax=ax3
)

ax3.set_xlabel('Keep Ratio (Fraction of Tokens Kept)')
ax3.set_ylabel('Average METEOR Score')
ax3.set_xticks(df_plot['Keep Ratio (Value)'].unique())
ax3.set_xticklabels([f"{int(r*100)}%" for r in df_plot['Keep Ratio (Value)'].unique()])

plt.tight_layout()

# Display all figures
plt.show()

NameError: name 'df_final_table' is not defined